## 3 Programming a neural network

In [1]:
import numpy as np
from sklearn import datasets

In [2]:
class ReLULayer(object):
    def forward(self, input):
        # remember the input for later backpropagation
        self.input = input
        # return the ReLU of the input
        relu = np.maximum(0, input) # keep positive values and set negative values to zero.
        return relu

    def backward(self, upstream_gradient):
        # compute the derivative of ReLU from upstream_gradient and the stored input
        # derivative of ReLU is 1 for input > 0 and 0 for input <= 0. so, gradients are passed through only where the sotred input was positive.
        downstream_gradient = upstream_gradient * (self.input > 0)
        return downstream_gradient

    def update(self, learning_rate):
        pass # ReLU is parameter-free

In [3]:
class OutputLayer(object):
    def __init__(self, n_classes):
        self.n_classes = n_classes

    def forward(self, input):
        # remember the input for later backpropagation
        self.input = input
        # return the softmax of the input
        softmax = np.exp(input - np.max(input, axis=1, keepdims=True)) / np.sum(np.exp(input - np.max(input, axis=1, keepdims=True)), axis=1, keepdims=True)
        return softmax

    def backward(self, predicted_posteriors, true_labels):
        # return the loss derivative with respect to the stored inputs
        # (use cross-entropy loss and the chain rule for softmax,
        #  as derived in the lecture)
        batch_size = predicted_posteriors.shape[0]

        downstream_gradient = predicted_posteriors.copy()
        downstream_gradient[np.arange(batch_size), true_labels] -= 1

        downstream_gradient /= batch_size
        return downstream_gradient

    def update(self, learning_rate):
        pass # softmax is parameter-free

In [4]:
class LinearLayer(object):
    def __init__(self, n_inputs, n_outputs):
        self.n_inputs  = n_inputs
        self.n_outputs = n_outputs
        # randomly initialize weights and intercepts
        self.B = np.random.normal(loc=0.0, scale=0.1, size=(n_inputs, n_outputs))
        self.b = np.random.normal(loc=0.0, scale=0.1, size=(n_outputs,))

    def forward(self, input):
        # remember the input for later backpropagation
        self.input = input
        # compute the scalar product of input and weights
        # (these are the preactivations for the subsequent non-linear layer)
        preactivations = input @ self.B + self.b
        return preactivations

    def backward(self, upstream_gradient):
        # compute the derivative of the weights from
        # upstream_gradient and the stored input
        self.grad_b = np.sum(upstream_gradient, axis=0)
        self.grad_B = self.input.T @ upstream_gradient
        # compute the downstream gradient to be passed to the preceding layer
        downstream_gradient = upstream_gradient @ self.B.T
        return downstream_gradient

    def update(self, learning_rate):
        # update the weights by batch gradient descent
        self.B = self.B - learning_rate * self.grad_B
        self.b = self.b - learning_rate * self.grad_b

In [5]:
class MLP(object):
    def __init__(self, n_features, layer_sizes):
        # constuct a multi-layer perceptron
        # with ReLU activation in the hidden layers and softmax output
        # (i.e. it predicts the posterior probability of a classification problem)
        #
        # n_features: number of inputs
        # len(layer_size): number of layers
        # layer_size[k]: number of neurons in layer k
        # (specifically: layer_sizes[-1] is the number of classes)
        self.n_layers = len(layer_sizes)
        self.layers   = []

        # create interior layers (linear + ReLU)
        n_in = n_features
        for n_out in layer_sizes[:-1]:
            self.layers.append(LinearLayer(n_in, n_out))
            self.layers.append(ReLULayer())
            n_in = n_out

        # create last linear layer + output layer
        n_out = layer_sizes[-1]
        self.layers.append(LinearLayer(n_in, n_out))
        self.layers.append(OutputLayer(n_out))

    def forward(self, X):
        # X is a mini-batch of instances
        batch_size = X.shape[0]
        # flatten the other dimensions of X (in case instances are images)
        X = X.reshape(batch_size, -1)

        # compute the forward pass
        # (implicitly stores internal activations for later backpropagation)
        result = X
        for layer in self.layers:
            result = layer.forward(result)
        return result

    def backward(self, predicted_posteriors, true_classes):
        # perform backpropagation w.r.t. the prediction for the latest mini-batch X
        gradient = self.layers[-1].backward(predicted_posteriors, true_classes)

        for layer in reversed(self.layers[:-1]):
            gradient = layer.backward(gradient)

    def update(self, X, Y, learning_rate):
        posteriors = self.forward(X)
        self.backward(posteriors, Y)
        for layer in self.layers:
            layer.update(learning_rate)

    def train(self, x, y, n_epochs, batch_size, learning_rate):
        N = len(x)
        n_batches = N // batch_size
        for i in range(n_epochs):
            # print("Epoch", i)
            # reorder data for every epoch
            # (i.e. sample mini-batches without replacement)
            permutation = np.random.permutation(N)

            for batch in range(n_batches):
                # create mini-batch
                start = batch * batch_size
                x_batch = x[permutation[start:start+batch_size]]
                y_batch = y[permutation[start:start+batch_size]]

                # perform one forward and backward pass and update network parameters
                self.update(x_batch, y_batch, learning_rate)

In [6]:
if __name__=="__main__":

    # set training/test set size
    N = 2000

    # create training and test data
    X_train, Y_train = datasets.make_moons(N, noise=0.05)
    X_test,  Y_test  = datasets.make_moons(N, noise=0.05)
    n_features = 2
    n_classes  = 2

    # standardize features to be in [-1, 1]
    offset  = X_train.min(axis=0)
    scaling = X_train.max(axis=0) - offset
    X_train = ((X_train - offset) / scaling - 0.5) * 2.0
    X_test  = ((X_test  - offset) / scaling - 0.5) * 2.0

    # set hyperparameters (play with these!)
    n_epochs = 100
    batch_size = 200
    learning_rate = 0.05
    
    layer_sizes_list = [
        [2, 2, n_classes],
        [3, 3, n_classes],
        [5, 5, n_classes],
        [30, 30, n_classes]
    ]

    for layer_sizes in layer_sizes_list:

        # create network
        network = MLP(n_features, layer_sizes)

        # train
        network.train(X_train, Y_train, n_epochs, batch_size, learning_rate)

        # test
        predicted_posteriors = network.forward(X_test)
        
        # determine class predictions from posteriors by winner-takes-all rule
        predicted_classes = np.argmax(predicted_posteriors, axis=1) # select the class with the highest probability for each test sample
        
        # compute and output the error rate of predicted_classes
        error_rate = np.mean(predicted_classes != Y_test) # the error rate is the fraction of test samples whose predicted class is different from the true label
        
        print("network:", layer_sizes)
        print("error rate:", error_rate)
        print()

network: [2, 2, 2]
error rate: 0.116

network: [3, 3, 2]
error rate: 0.1105

network: [5, 5, 2]
error rate: 0.111

network: [30, 30, 2]
error rate: 0.1095



I increased the epoch from 5 to 100 for the test. When the number of neurons is increased, the error rate genrally decreases. The smallest network [2, 2, 2] gives the highest error rate, 0.1365. The networks [3, 3, 2] and [5, 5, 2] perform slightly better, with error rates around 0.116 and 0.1145. The largest network [30, 30, 2] gives the best result, with an error rate of 0.1. This makes sense because the make_moons dataset is not linearly separable. Larger hidden layers can model a more flexible nonlinear decision boundary, so they can classify the two moon-shaped classes better. However, the improvement from [3, 3, 2] to [5, 5, 2] is small, which suggest that increasing the network size does not always give a large improvement.